In [ ]:
# ============================================================
# PHYSICS-GUIDED PETROLEUM PRODUCT ADULTERATION DATASET
# FINAL RESEARCH VERSION
# ============================================================
#
# Study:
# Development of an Edge AI Framework for Real-Time Detection
# of Petroleum Products Adulteration
#
# Dataset purpose:
# Physics-guided simulated training dataset
#
# IMPORTANT:
# - This dataset is simulated.
# - Laboratory measurements are treated as independent validation data.
# - No laboratory file is used in this simulation code.
# - No Label column is created.
# - Row order is randomized after generation.
#
# Final dataset size:
# 216,080 observations
#
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import json
import math
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# ============================================================
# 2. GLOBAL SETTINGS
# ============================================================

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Reproducibility
rng = np.random.default_rng(RANDOM_SEED)

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

OUTPUT_ROOT = Path("/content/physics_guided_petroleum_dataset")

DATASET_DIR = OUTPUT_ROOT / "01_Dataset"
PLOTS_DIR = OUTPUT_ROOT / "02_Plots"
REPORTS_DIR = OUTPUT_ROOT / "03_Reports"
DOCS_DIR = OUTPUT_ROOT / "04_Documentation"

for folder in [
    OUTPUT_ROOT,
    DATASET_DIR,
    PLOTS_DIR,
    REPORTS_DIR,
    DOCS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# 3. PROFESSIONAL PLOT SETTINGS
# ============================================================

sns.set_theme(
    style="whitegrid",
    context="paper"
)

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,

    "font.family": "DejaVu Sans",

    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 11,

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,

    "legend.fontsize": 8.5,
    "legend.title_fontsize": 9,

    "axes.spines.top": False,
    "axes.spines.right": False,

    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    "axes.titleweight": "bold"
})


# ============================================================
# 4. RESEARCH-QUALITY COLOR PALETTE
# ============================================================

# Color-blind-friendly palette

FUEL_COLORS = {
    "PMS": "#0072B2",   # blue
    "AGO": "#E69F00",   # orange
    "DPK": "#009E73"    # green
}

ADULTERANT_COLORS = {
    "Kerosene": "#D55E00",
    "AGO": "#E69F00",
    "Water": "#56B4E9",
    "Waste_Engine_Oil": "#7F3C8D",
    "Methanol": "#CC79A7",
    "Ethanol": "#009E73"
}


# ============================================================
# 5. EXPERIMENTAL DESIGN
# ============================================================

TOTAL_VOLUME_ML = 200

# Concentration levels follow the laboratory experimental design
ADULTERATION_LEVELS = list(range(0, 51, 5))

NONZERO_LEVELS = list(range(5, 51, 5))


# ------------------------------------------------------------
# Base fuels
# ------------------------------------------------------------

BASE_FUELS = [
    "PMS",
    "AGO",
    "DPK"
]


# ------------------------------------------------------------
# Approved fuel-adulterant combinations
# ------------------------------------------------------------
#
# PMS:
#   Kerosene
#   AGO
#   Water
#   Waste Engine Oil
#   Methanol
#   Ethanol
#
# AGO:
#   Kerosene
#   Water
#   Waste Engine Oil
#   Methanol
#   Ethanol
#
# DPK:
#   AGO
#   Water
#   Waste Engine Oil
#   Methanol
#   Ethanol
#
# Excluded:
#   AGO + AGO
#   DPK + Kerosene
#
# Total valid pairs = 16
# ------------------------------------------------------------

VALID_PAIRS = [
    ("PMS", "Kerosene"),
    ("PMS", "AGO"),
    ("PMS", "Water"),
    ("PMS", "Waste_Engine_Oil"),
    ("PMS", "Methanol"),
    ("PMS", "Ethanol"),

    ("AGO", "Kerosene"),
    ("AGO", "Water"),
    ("AGO", "Waste_Engine_Oil"),
    ("AGO", "Methanol"),
    ("AGO", "Ethanol"),

    ("DPK", "AGO"),
    ("DPK", "Water"),
    ("DPK", "Waste_Engine_Oil"),
    ("DPK", "Methanol"),
    ("DPK", "Ethanol")
]


# ============================================================
# 6. DATASET SIZE DESIGN
# ============================================================

# Pure observations:
#
# 25,360 observations × 3 pure fuels
# = 76,080 pure observations
#
# Adulterated:
#
# 16 fuel-adulterant pairs
# × 10 nonzero concentration levels
# × 875 observations
# = 140,000 adulterated observations
#
# Total:
#
# 76,080 + 140,000 = 216,080
# ============================================================

PURE_OBSERVATIONS_PER_FUEL = 25360

ADULTERATED_OBSERVATIONS_PER_CONDITION = 875

EXPECTED_PURE = (
    len(BASE_FUELS)
    * PURE_OBSERVATIONS_PER_FUEL
)

EXPECTED_ADULTERATED = (
    len(VALID_PAIRS)
    * len(NONZERO_LEVELS)
    * ADULTERATED_OBSERVATIONS_PER_CONDITION
)

EXPECTED_TOTAL = EXPECTED_PURE + EXPECTED_ADULTERATED

assert EXPECTED_TOTAL == 216080


# ============================================================
# 7. PURE FUEL / ADULTERANT PROPERTY DATABASE
# ============================================================
#
# These values are simulation reference parameters.
#
# They are NOT presented as laboratory measurements.
#
# Density:
#   g/mL
#
# Viscosity:
#   cP
#
# Refractive index:
#   dimensionless
#
# Dielectric constant:
#   dimensionless
#
# Colour:
#   PMS/DPK -> Saybolt
#   AGO     -> ASTM Color
# ============================================================

PROPERTY_DATABASE = {

    "PMS": {
        "density": 0.740,
        "viscosity": 0.60,
        "refractive_index": 1.4200,
        "dielectric": 2.10,
        "colour": 25.0,
        "colour_scale": "Saybolt"
    },

    "AGO": {
        "density": 0.840,
        "viscosity": 2.50,
        "refractive_index": 1.4600,
        "dielectric": 3.20,
        "colour": 3.00,
        "colour_scale": "ASTM"
    },

    "DPK": {
        "density": 0.800,
        "viscosity": 1.80,
        "refractive_index": 1.4400,
        "dielectric": 2.00,
        "colour": 22.0,
        "colour_scale": "Saybolt"
    },

    "Kerosene": {
        "density": 0.800,
        "viscosity": 1.80,
        "refractive_index": 1.4400,
        "dielectric": 2.00
    },

    "Water": {
        "density": 0.998,
        "viscosity": 1.00,
        "refractive_index": 1.3330,
        "dielectric": 78.5
    },

    "Waste_Engine_Oil": {
        "density": 0.880,
        "viscosity": 100.0,
        "refractive_index": 1.4800,
        "dielectric": 3.50
    },

    "Methanol": {
        "density": 0.792,
        "viscosity": 0.55,
        "refractive_index": 1.3290,
        "dielectric": 32.6
    },

    "Ethanol": {
        "density": 0.789,
        "viscosity": 1.07,
        "refractive_index": 1.3610,
        "dielectric": 24.6
    }
}


# ============================================================
# 8. COLOUR RESPONSE MODEL
# ============================================================
#
# Colour is treated as an empirical response variable rather
# than forcing it into the same physical mixing equation used
# for density, viscosity, refractive index and dielectric
# constant.
#
# Values are simulation response endpoints.
# ============================================================

COLOUR_RESPONSE = {

    "PMS": {
        "Kerosene": (25.0, 22.0),
        "AGO": (25.0, 14.0),
        "Water": (25.0, 15.0),
        "Waste_Engine_Oil": (25.0, 13.0),
        "Methanol": (25.0, 27.0),
        "Ethanol": (25.0, 28.0)
    },

    "AGO": {
        "Kerosene": (3.0, 5.0),
        "Water": (3.0, 5.0),
        "Waste_Engine_Oil": (3.0, 5.5),
        "Methanol": (3.0, 2.0),
        "Ethanol": (3.0, 2.0)
    },

    "DPK": {
        "AGO": (22.0, 26.0),
        "Water": (22.0, 12.0),
        "Waste_Engine_Oil": (22.0, 15.0),
        "Methanol": (22.0, 26.0),
        "Ethanol": (22.0, 26.0)
    }
}


# ============================================================
# 9. GAUSSIAN SENSOR / MEASUREMENT DISTURBANCE
# ============================================================
#
# X_sim = X_model × (1 + epsilon)
#
# epsilon ~ N(0, sigma)
#
# These are simulation assumptions rather than claimed
# laboratory uncertainty values.
#
# Concentration and mixture volumes are NOT disturbed.
# ============================================================

NOISE_RSD = {
    "density": 0.0020,             # 0.20%
    "viscosity": 0.0100,           # 1.00%
    "refractive_index": 0.0005,    # 0.05%
    "dielectric": 0.0150,          # 1.50%
    "colour": 0.0100               # 1.00%
}


# ============================================================
# 10. HELPER FUNCTIONS
# ============================================================

def apply_relative_noise(value, rsd, rng):
    """
    Apply multiplicative Gaussian measurement disturbance.

    X_sim = X_model * (1 + epsilon)
    epsilon ~ N(0, rsd)
    """

    noisy_value = value * (
        1.0 + rng.normal(0.0, rsd)
    )

    return noisy_value


def volume_fractions(adulteration_percent):
    """
    Calculate exact volume fractions for a 200 mL mixture.
    """

    adulterant_fraction = adulteration_percent / 100.0
    base_fraction = 1.0 - adulterant_fraction

    return base_fraction, adulterant_fraction


def linear_mixture(base_value, adulterant_value, phi_base, phi_adulterant):
    """
    Linear volume-fraction mixing model.
    """

    return (
        phi_base * base_value
        + phi_adulterant * adulterant_value
    )


def geometric_mixture(base_value, adulterant_value,
                      phi_base, phi_adulterant):
    """
    Logarithmic / geometric mixing model.

    Suitable for properties represented on a multiplicative
    scale such as viscosity and dielectric constant.
    """

    base_value = max(base_value, 1e-12)
    adulterant_value = max(adulterant_value, 1e-12)

    log_mix = (
        phi_base * np.log(base_value)
        + phi_adulterant * np.log(adulterant_value)
    )

    return np.exp(log_mix)


def lorentz_lorenz_mix(n_base, n_adulterant,
                       phi_base, phi_adulterant):
    """
    Lorentz-Lorenz volume-fraction mixing model for
    refractive index.
    """

    def lorentz_lorenz(n):
        return (
            (n ** 2 - 1.0)
            / (n ** 2 + 2.0)
        )

    r_base = lorentz_lorenz(n_base)
    r_adulterant = lorentz_lorenz(n_adulterant)

    r_mix = (
        phi_base * r_base
        + phi_adulterant * r_adulterant
    )

    denominator = 1.0 - r_mix

    n_mix_squared = (
        1.0 + 2.0 * r_mix
    ) / denominator

    return np.sqrt(n_mix_squared)


def colour_mix(base_fuel, adulterant, concentration):
    """
    Controlled empirical colour-response model.

    Colour is interpolated between the base-fuel colour and
    the response endpoint for the selected adulterant.
    """

    base_colour, endpoint_colour = COLOUR_RESPONSE[
        base_fuel
    ][adulterant]

    fraction = concentration / 50.0

    response = (
        base_colour
        + fraction
        * (endpoint_colour - base_colour)
    )

    return response


def calculate_properties(base_fuel,
                         adulterant,
                         concentration,
                         rng):
    """
    Calculate all simulated physical properties.
    """

    base = PROPERTY_DATABASE[base_fuel]
    adulterant_data = PROPERTY_DATABASE[adulterant]

    phi_base, phi_adulterant = volume_fractions(
        concentration
    )

    # --------------------------------------------------------
    # Density
    # --------------------------------------------------------

    density_model = linear_mixture(
        base["density"],
        adulterant_data["density"],
        phi_base,
        phi_adulterant
    )

    # --------------------------------------------------------
    # Viscosity
    # --------------------------------------------------------

    viscosity_model = geometric_mixture(
        base["viscosity"],
        adulterant_data["viscosity"],
        phi_base,
        phi_adulterant
    )

    # --------------------------------------------------------
    # Refractive index
    # --------------------------------------------------------

    refractive_index_model = lorentz_lorenz_mix(
        base["refractive_index"],
        adulterant_data["refractive_index"],
        phi_base,
        phi_adulterant
    )

    # --------------------------------------------------------
    # Dielectric constant
    # --------------------------------------------------------

    dielectric_model = geometric_mixture(
        base["dielectric"],
        adulterant_data["dielectric"],
        phi_base,
        phi_adulterant
    )

    # --------------------------------------------------------
    # Colour
    # --------------------------------------------------------

    colour_model = colour_mix(
        base_fuel,
        adulterant,
        concentration
    )

    # --------------------------------------------------------
    # Apply Gaussian measurement/sensor disturbance
    # --------------------------------------------------------

    density = apply_relative_noise(
        density_model,
        NOISE_RSD["density"],
        rng
    )

    viscosity = apply_relative_noise(
        viscosity_model,
        NOISE_RSD["viscosity"],
        rng
    )

    refractive_index = apply_relative_noise(
        refractive_index_model,
        NOISE_RSD["refractive_index"],
        rng
    )

    dielectric = apply_relative_noise(
        dielectric_model,
        NOISE_RSD["dielectric"],
        rng
    )

    colour = apply_relative_noise(
        colour_model,
        NOISE_RSD["colour"],
        rng
    )

    # --------------------------------------------------------
    # Physical safety checks
    # --------------------------------------------------------

    density = max(density, 0.001)
    viscosity = max(viscosity, 0.001)
    refractive_index = max(refractive_index, 1.0)
    dielectric = max(dielectric, 0.001)
    colour = max(colour, 0.001)

    return {
        "Density_g_mL": density,
        "Viscosity_cP": viscosity,
        "Refractive_Index": refractive_index,
        "Dielectric_Constant": dielectric,
        "Colour_Value": colour
    }


# ============================================================
# 11. GENERATE PURE FUEL OBSERVATIONS
# ============================================================

print("=" * 70)
print("GENERATING PHYSICS-GUIDED PETROLEUM DATASET")
print("=" * 70)

print("\nGenerating pure-fuel observations...")

rows = []

for fuel in BASE_FUELS:

    base = PROPERTY_DATABASE[fuel]

    for _ in range(PURE_OBSERVATIONS_PER_FUEL):

        # Pure fuel has no adulterant.
        concentration = 0

        density = apply_relative_noise(
            base["density"],
            NOISE_RSD["density"],
            rng
        )

        viscosity = apply_relative_noise(
            base["viscosity"],
            NOISE_RSD["viscosity"],
            rng
        )

        refractive_index = apply_relative_noise(
            base["refractive_index"],
            NOISE_RSD["refractive_index"],
            rng
        )

        dielectric = apply_relative_noise(
            base["dielectric"],
            NOISE_RSD["dielectric"],
            rng
        )

        colour = apply_relative_noise(
            base["colour"],
            NOISE_RSD["colour"],
            rng
        )

        rows.append({
            "Base_Fuel": fuel,
            "Adulterant": "None",
            "Adulteration_%": concentration,

            "Base_Volume_mL": TOTAL_VOLUME_ML,
            "Adulterant_Volume_mL": 0,

            "Density_g_mL": max(density, 0.001),
            "Viscosity_cP": max(viscosity, 0.001),
            "Refractive_Index": max(
                refractive_index,
                1.0
            ),
            "Dielectric_Constant": max(
                dielectric,
                0.001
            ),
            "Colour_Value": max(
                colour,
                0.001
            )
        })


print(
    f"Pure observations generated: "
    f"{len(rows):,}"
)


# ============================================================
# 12. GENERATE ADULTERATED OBSERVATIONS
# ============================================================

print("\nGenerating adulterated observations...")

adulterated_start = len(rows)

for base_fuel, adulterant in VALID_PAIRS:

    for concentration in NONZERO_LEVELS:

        # Exact mixture volumes
        adulterant_volume = (
            TOTAL_VOLUME_ML
            * concentration
            / 100.0
        )

        base_volume = (
            TOTAL_VOLUME_ML
            - adulterant_volume
        )

        for _ in range(
            ADULTERATED_OBSERVATIONS_PER_CONDITION
        ):

            properties = calculate_properties(
                base_fuel=base_fuel,
                adulterant=adulterant,
                concentration=concentration,
                rng=rng
            )

            rows.append({
                "Base_Fuel": base_fuel,
                "Adulterant": adulterant,
                "Adulteration_%": concentration,

                "Base_Volume_mL": base_volume,
                "Adulterant_Volume_mL": adulterant_volume,

                "Density_g_mL":
                    properties["Density_g_mL"],

                "Viscosity_cP":
                    properties["Viscosity_cP"],

                "Refractive_Index":
                    properties["Refractive_Index"],

                "Dielectric_Constant":
                    properties["Dielectric_Constant"],

                "Colour_Value":
                    properties["Colour_Value"]
            })


adulterated_generated = len(rows) - adulterated_start

print(
    f"Adulterated observations generated: "
    f"{adulterated_generated:,}"
)


# ============================================================
# 13. CREATE DATAFRAME
# ============================================================

dataset = pd.DataFrame(rows)


# ============================================================
# 14. RANDOMIZE ROW ORDER
# ============================================================
#
# IMPORTANT:
#
# The values are NOT mixed between conditions.
# Only the final row order is randomized.
#
# Sample_ID is assigned AFTER randomization.
#
# Therefore:
#   Sample 1, 2, 3, ... do not reveal the experimental block.
# ============================================================

print("\nRandomizing final row order...")

dataset = dataset.sample(
    frac=1.0,
    random_state=RANDOM_SEED
).reset_index(drop=True)


# ------------------------------------------------------------
# Assign Sample_ID only after randomization
# ------------------------------------------------------------

dataset.insert(
    0,
    "Sample_ID",
    np.arange(
        1,
        len(dataset) + 1
    )
)


# ============================================================
# 15. FINAL COLUMN ORDER
# ============================================================

FINAL_COLUMNS = [
    "Sample_ID",
    "Base_Fuel",
    "Adulterant",
    "Adulteration_%",
    "Base_Volume_mL",
    "Adulterant_Volume_mL",
    "Density_g_mL",
    "Viscosity_cP",
    "Refractive_Index",
    "Dielectric_Constant",
    "Colour_Value"
]

dataset = dataset[FINAL_COLUMNS]


# ============================================================
# 16. DATA TYPES
# ============================================================

dataset["Sample_ID"] = (
    dataset["Sample_ID"]
    .astype(int)
)

dataset["Adulteration_%"] = (
    dataset["Adulteration_%"]
    .astype(int)
)

dataset["Base_Volume_mL"] = (
    dataset["Base_Volume_mL"]
    .round(2)
)

dataset["Adulterant_Volume_mL"] = (
    dataset["Adulterant_Volume_mL"]
    .round(2)
)

dataset["Density_g_mL"] = (
    dataset["Density_g_mL"]
    .round(6)
)

dataset["Viscosity_cP"] = (
    dataset["Viscosity_cP"]
    .round(6)
)

dataset["Refractive_Index"] = (
    dataset["Refractive_Index"]
    .round(6)
)

dataset["Dielectric_Constant"] = (
    dataset["Dielectric_Constant"]
    .round(6)
)

dataset["Colour_Value"] = (
    dataset["Colour_Value"]
    .round(6)
)


# ============================================================
# 17. INTEGRITY VALIDATION
# ============================================================

print("\nPerforming dataset integrity checks...")

# Total rows
assert len(dataset) == 216080

# Unique Sample_ID
assert dataset["Sample_ID"].is_unique

# Sample_ID sequence
assert (
    dataset["Sample_ID"].min() == 1
)

assert (
    dataset["Sample_ID"].max()
    == len(dataset)
)

# No missing values
assert dataset.isna().sum().sum() == 0

# Correct base fuels
assert set(
    dataset["Base_Fuel"].unique()
) == set(BASE_FUELS)

# Pure records
pure_mask = (
    dataset["Adulterant"] == "None"
)

assert (
    dataset.loc[
        pure_mask,
        "Adulteration_%"
    ] == 0
).all()

assert (
    dataset.loc[
        pure_mask,
        "Adulterant_Volume_mL"
    ] == 0
).all()

assert (
    dataset.loc[
        pure_mask,
        "Base_Volume_mL"
    ] == TOTAL_VOLUME_ML
).all()

# Adulterated records
adulterated_mask = (
    dataset["Adulterant"] != "None"
)

assert (
    dataset.loc[
        adulterated_mask,
        "Adulteration_%"
    ].isin(NONZERO_LEVELS)
).all()

# Volume conservation
volume_total = (
    dataset["Base_Volume_mL"]
    + dataset["Adulterant_Volume_mL"]
)

assert np.allclose(
    volume_total,
    TOTAL_VOLUME_ML,
    atol=1e-6
)

# No excluded combinations
assert not (
    (
        dataset["Base_Fuel"] == "AGO"
    )
    &
    (
        dataset["Adulterant"] == "AGO"
    )
).any()

assert not (
    (
        dataset["Base_Fuel"] == "DPK"
    )
    &
    (
        dataset["Adulterant"] == "Kerosene"
    )
).any()

# Positive physical values
for column in [
    "Density_g_mL",
    "Viscosity_cP",
    "Refractive_Index",
    "Dielectric_Constant",
    "Colour_Value"
]:

    assert (
        dataset[column] > 0
    ).all()


print("All integrity checks passed.")


# ============================================================
# 18. DATASET SUMMARY
# ============================================================

pure_count = (
    dataset["Adulterant"] == "None"
).sum()

adulterated_count = (
    dataset["Adulterant"] != "None"
).sum()

print("\n" + "=" * 70)
print("FINAL DATASET SUMMARY")
print("=" * 70)

print(
    f"Total observations : {len(dataset):,}"
)

print(
    f"Pure observations  : {pure_count:,}"
)

print(
    f"Adulterated        : {adulterated_count:,}"
)

print(
    f"Base fuels         : "
    f"{dataset['Base_Fuel'].nunique()}"
)

print(
    f"Adulterants        : "
    f"{dataset.loc[adulterated_mask, 'Adulterant'].nunique()}"
)

print(
    f"Concentration levels: "
    f"{dataset['Adulteration_%'].nunique()}"
)


# ============================================================
# 19. SAVE RAW CSV
# ============================================================

csv_file = (
    DATASET_DIR
    / "physics_guided_petroleum_adulteration_raw_dataset.csv"
)

dataset.to_csv(
    csv_file,
    index=False
)

print(
    f"\nCSV saved:\n{csv_file}"
)


# ============================================================
# 20. SAVE EXCEL WORKBOOK
# ============================================================

excel_file = (
    DATASET_DIR
    / "physics_guided_petroleum_adulteration_raw_dataset.xlsx"
)

with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    dataset.to_excel(
        writer,
        sheet_name="Raw_Dataset",
        index=False
    )

    # Design frequency
    condition_summary = (
        dataset
        .groupby(
            [
                "Base_Fuel",
                "Adulterant",
                "Adulteration_%"
            ]
        )
        .size()
        .reset_index(
            name="Observation_Count"
        )
    )

    condition_summary.to_excel(
        writer,
        sheet_name="Design_Frequency",
        index=False
    )

    # Fuel summary
    fuel_summary = (
        dataset
        .groupby("Base_Fuel")
        .size()
        .reset_index(
            name="Observation_Count"
        )
    )

    fuel_summary.to_excel(
        writer,
        sheet_name="Base_Fuel_Summary",
        index=False
    )

    # Adulterant summary
    adulterant_summary = (
        dataset
        .groupby("Adulterant")
        .size()
        .reset_index(
            name="Observation_Count"
        )
    )

    adulterant_summary.to_excel(
        writer,
        sheet_name="Adulterant_Summary",
        index=False
    )

    # Concentration summary
    concentration_summary = (
        dataset
        .groupby("Adulteration_%")
        .size()
        .reset_index(
            name="Observation_Count"
        )
    )

    concentration_summary.to_excel(
        writer,
        sheet_name="Concentration_Summary",
        index=False
    )


print(
    f"Excel workbook saved:\n{excel_file}"
)


# ============================================================
# 21. CREATE METADATA
# ============================================================

metadata = {

    "dataset_name":
        "Physics-Guided Petroleum Adulteration Raw Dataset",

    "purpose":
        "Simulated training dataset for Edge AI research on petroleum product adulteration detection",

    "dataset_type":
        "Physics-guided simulated dataset",

    "validation_strategy":
        "Independent laboratory measurements",

    "total_observations":
        int(len(dataset)),

    "pure_observations":
        int(pure_count),

    "adulterated_observations":
        int(adulterated_count),

    "base_fuels":
        BASE_FUELS,

    "valid_pairs":
        [
            {
                "base_fuel": base,
                "adulterant": adulterant
            }
            for base, adulterant
            in VALID_PAIRS
        ],

    "excluded_pairs": [
        "AGO + AGO",
        "DPK + Kerosene"
    ],

    "adulteration_levels_percent":
        ADULTERATION_LEVELS,

    "total_mixture_volume_mL":
        TOTAL_VOLUME_ML,

    "pure_observations_per_fuel":
        PURE_OBSERVATIONS_PER_FUEL,

    "adulterated_observations_per_condition":
        ADULTERATED_OBSERVATIONS_PER_CONDITION,

    "random_seed":
        RANDOM_SEED,

    "row_order":
        "Randomized after dataset generation",

    "sample_id_assignment":
        "Assigned after randomization",

    "noise_model":
        "Multiplicative Gaussian relative disturbance",

    "noise_formula":
        "X_sim = X_model * (1 + epsilon), epsilon ~ N(0, sigma)",

    "noise_relative_standard_deviations":
        NOISE_RSD,

    "mixing_models": {
        "density":
            "Linear volume-fraction mixing",

        "viscosity":
            "Logarithmic/geometric mixing",

        "refractive_index":
            "Lorentz-Lorenz volume-fraction mixing",

        "dielectric_constant":
            "Logarithmic/geometric mixing",

        "colour":
            "Controlled empirical response curve"
    },

    "raw_columns":
        FINAL_COLUMNS,

    "important_note":
        "The simulated dataset is not laboratory data. Laboratory measurements are maintained as an independent validation dataset."
}


metadata_file = (
    DOCS_DIR
    / "dataset_metadata.json"
)

with open(
    metadata_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )


# ============================================================
# 22. DESIGN REPORT
# ============================================================

design_report = f"""
PHYSICS-GUIDED PETROLEUM ADULTERATION DATASET
============================================================

Dataset type:
Physics-guided simulated raw training dataset

Total observations:
{len(dataset):,}

Pure observations:
{pure_count:,}

Adulterated observations:
{adulterated_count:,}

Base fuels:
PMS, AGO, DPK

Adulteration levels:
0%, 5%, 10%, 15%, 20%, 25%, 30%, 35%, 40%, 45%, 50%

Total mixture volume:
200 mL

Valid fuel-adulterant combinations:
16

Excluded combinations:
AGO + AGO
DPK + Kerosene

Pure observations:
25,360 observations per base fuel

Adulterated observations:
875 observations per fuel-adulterant-concentration condition

Randomization:
The generated records were randomly reordered before Sample_ID
was assigned. This prevents sequential Sample_ID values from
revealing experimental condition blocks.

Physical-property models:
1. Density:
   Linear volume-fraction mixing

2. Viscosity:
   Logarithmic/geometric mixing

3. Refractive index:
   Lorentz-Lorenz volume-fraction mixing

4. Dielectric constant:
   Logarithmic/geometric mixing

5. Colour:
   Controlled empirical response curves

Gaussian disturbance:
Multiplicative Gaussian disturbance was applied only to the
simulated physical-property measurements. Adulteration
concentration and mixture volumes were kept exact.

Important scientific distinction:
This dataset is simulated and is intended for model development.
Actual laboratory measurements are treated separately as
independent validation data.

No Label column:
The dataset is intentionally retained as a raw dataset.
Target variables and preprocessing are to be constructed later.

============================================================
"""


design_report_file = (
    DOCS_DIR
    / "dataset_design_report.txt"
)

with open(
    design_report_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(design_report)


# ============================================================
# 23. PLOT SAVING FUNCTION
# ============================================================

def save_research_figure(
    fig,
    filename,
    dpi=600
):

    png_path = PLOTS_DIR / f"{filename}.png"
    pdf_path = PLOTS_DIR / f"{filename}.pdf"

    fig.savefig(
        png_path,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    return png_path, pdf_path


# ============================================================
# 24. FIGURE 1 — PURE VS ADULTERATED COMPOSITION
# ============================================================

composition_data = pd.DataFrame({
    "Category": [
        "Pure",
        "Adulterated"
    ],
    "Count": [
        pure_count,
        adulterated_count
    ]
})

fig, ax = plt.subplots(
    figsize=(8.5, 5.5)
)

bars = ax.bar(
    composition_data["Category"],
    composition_data["Count"],
    width=0.55,
    color=[
        "#0072B2",
        "#D55E00"
    ]
)

ax.set_title(
    "Dataset Composition: Pure and Adulterated Samples"
)

ax.set_ylabel(
    "Number of observations"
)

ax.set_xlabel(
    "Sample category"
)

ax.set_ylim(
    0,
    max(composition_data["Count"]) * 1.15
)

for bar, count in zip(
    bars,
    composition_data["Count"]
):

    percentage = (
        count
        / len(dataset)
        * 100
    )

    ax.text(
        bar.get_x()
        + bar.get_width() / 2,
        bar.get_height()
        + max(composition_data["Count"]) * 0.02,
        f"{count:,}\n({percentage:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )

sns.despine(ax=ax)

save_research_figure(
    fig,
    "Figure_01_Dataset_Composition"
)


# ============================================================
# 25. FIGURE 2 — BASE-FUEL DISTRIBUTION
# ============================================================

fuel_counts = (
    dataset["Base_Fuel"]
    .value_counts()
    .reindex(BASE_FUELS)
)

fig, ax = plt.subplots(
    figsize=(8.5, 5.5)
)

bars = ax.barh(
    fuel_counts.index,
    fuel_counts.values,
    color=[
        FUEL_COLORS[fuel]
        for fuel in fuel_counts.index
    ],
    height=0.55
)

ax.set_title(
    "Base-Fuel Distribution"
)

ax.set_xlabel(
    "Number of observations"
)

ax.set_ylabel(
    "Base fuel"
)

ax.invert_yaxis()

for bar, count in zip(
    bars,
    fuel_counts.values
):

    percentage = (
        count
        / len(dataset)
        * 100
    )

    ax.text(
        bar.get_width()
        + len(dataset) * 0.008,
        bar.get_y()
        + bar.get_height() / 2,
        f"{count:,} ({percentage:.1f}%)",
        va="center",
        fontsize=9
    )

ax.set_xlim(
    0,
    fuel_counts.max() * 1.15
)

sns.despine(ax=ax)

save_research_figure(
    fig,
    "Figure_02_Base_Fuel_Distribution"
)


# ============================================================
# 26. FIGURE 3 — ADULTERANT DISTRIBUTION
# ============================================================

adulterant_counts = (
    dataset.loc[
        adulterated_mask,
        "Adulterant"
    ]
    .value_counts()
)

adulterant_order = [
    "Kerosene",
    "AGO",
    "Water",
    "Waste_Engine_Oil",
    "Methanol",
    "Ethanol"
]

adulterant_counts = (
    adulterant_counts
    .reindex(adulterant_order)
)

fig, ax = plt.subplots(
    figsize=(9, 6)
)

bars = ax.barh(
    adulterant_counts.index,
    adulterant_counts.values,
    color=[
        ADULTERANT_COLORS[x]
        for x in adulterant_counts.index
    ],
    height=0.55
)

ax.set_title(
    "Adulterant Distribution in the Simulated Dataset"
)

ax.set_xlabel(
    "Number of observations"
)

ax.set_ylabel(
    "Adulterant"
)

ax.invert_yaxis()

for bar, count in zip(
    bars,
    adulterant_counts.values
):

    percentage = (
        count
        / adulterated_count
        * 100
    )

    ax.text(
        bar.get_width()
        + adulterated_count * 0.006,
        bar.get_y()
        + bar.get_height() / 2,
        f"{count:,} ({percentage:.1f}%)",
        va="center",
        fontsize=9
    )

ax.set_xlim(
    0,
    adulterant_counts.max() * 1.18
)

sns.despine(ax=ax)

save_research_figure(
    fig,
    "Figure_03_Adulterant_Distribution"
)


# ============================================================
# 27. FIGURE 4 — CONCENTRATION DISTRIBUTION
# ============================================================

concentration_counts = (
    dataset["Adulteration_%"]
    .value_counts()
    .sort_index()
)

fig, ax = plt.subplots(
    figsize=(9, 5.5)
)

bars = ax.bar(
    concentration_counts.index.astype(str),
    concentration_counts.values,
    width=0.72,
    color="#0072B2"
)

ax.set_title(
    "Distribution of Adulteration Concentration Levels"
)

ax.set_xlabel(
    "Adulteration concentration (%)"
)

ax.set_ylabel(
    "Number of observations"
)

for bar, count in zip(
    bars,
    concentration_counts.values
):

    ax.text(
        bar.get_x()
        + bar.get_width() / 2,
        bar.get_height()
        + concentration_counts.max() * 0.015,
        f"{count:,}",
        ha="center",
        va="bottom",
        fontsize=8
    )

ax.set_ylim(
    0,
    concentration_counts.max() * 1.15
)

sns.despine(ax=ax)

save_research_figure(
    fig,
    "Figure_04_Concentration_Distribution"
)


# ============================================================
# 28. FIGURE 5 — FEATURE DISTRIBUTIONS
# ============================================================

feature_columns = [
    "Density_g_mL",
    "Viscosity_cP",
    "Refractive_Index",
    "Dielectric_Constant",
    "Colour_Value"
]

feature_labels = {
    "Density_g_mL":
        "Density (g/mL)",

    "Viscosity_cP":
        "Viscosity (cP)",

    "Refractive_Index":
        "Refractive Index",

    "Dielectric_Constant":
        "Dielectric Constant",

    "Colour_Value":
        "Colour Value"
}

fig, axes = plt.subplots(
    3,
    2,
    figsize=(11, 12)
)

axes = axes.flatten()

for i, feature in enumerate(
    feature_columns
):

    ax = axes[i]

    sns.histplot(
        data=dataset,
        x=feature,
        bins=60,
        color="#0072B2",
        edgecolor="white",
        linewidth=0.25,
        ax=ax
    )

    ax.set_title(
        feature_labels[feature]
    )

    ax.set_xlabel(
        feature_labels[feature]
    )

    ax.set_ylabel(
        "Frequency"
    )

    sns.despine(ax=ax)

# Hide unused subplot
axes[-1].axis("off")

fig.suptitle(
    "Distributions of Simulated Physical Properties",
    fontsize=15,
    fontweight="bold",
    y=0.995
)

fig.tight_layout(
    rect=[0, 0, 1, 0.98]
)

save_research_figure(
    fig,
    "Figure_05_Feature_Distributions"
)


# ============================================================
# 29. FIGURE 6 — CORRELATION MATRIX
# ============================================================
#
# This is deliberately improved for research presentation:
#
# - large figure
# - Pearson correlation
# - diverging colour map
# - coefficients displayed
# - readable scientific variable names
# - square cells
# - clear colour bar
# - high-resolution output
# ============================================================

correlation_columns = [
    "Adulteration_%",
    "Density_g_mL",
    "Viscosity_cP",
    "Refractive_Index",
    "Dielectric_Constant",
    "Colour_Value"
]

correlation_labels = [
    "Adulteration (%)",
    "Density (g/mL)",
    "Viscosity (cP)",
    "Refractive Index",
    "Dielectric Constant",
    "Colour Value"
]

correlation_matrix = (
    dataset[correlation_columns]
    .corr(method="pearson")
)

correlation_matrix.index = correlation_labels
correlation_matrix.columns = correlation_labels

fig, ax = plt.subplots(
    figsize=(11, 9)
)

heatmap = sns.heatmap(
    correlation_matrix,
    ax=ax,

    cmap="RdBu_r",

    vmin=-1,
    vmax=1,
    center=0,

    annot=True,
    fmt=".2f",

    square=True,

    linewidths=0.7,
    linecolor="white",

    annot_kws={
        "fontsize": 10,
        "fontweight": "bold"
    },

    cbar_kws={
        "label": "Pearson correlation coefficient",
        "shrink": 0.82
    }
)

ax.set_title(
    "Pearson Correlation Matrix of Simulated Dataset Variables",
    fontsize=15,
    fontweight="bold",
    pad=18
)

ax.set_xlabel("")
ax.set_ylabel("")

plt.xticks(
    rotation=35,
    ha="right"
)

plt.yticks(
    rotation=0
)

fig.tight_layout()

save_research_figure(
    fig,
    "Figure_06_Pearson_Correlation_Matrix"
)


# ============================================================
# 30. FIGURE 7 — PROPERTY RESPONSE TO ADULTERATION
# ============================================================
#
# Three panels:
# PMS
# AGO
# DPK
#
# Each panel displays the mean response for each valid
# adulterant, with ±1 SD uncertainty bands.
# ============================================================

property_plot_definitions = [

    (
        "Density_g_mL",
        "Density (g/mL)",
        "Density Response to Adulteration"
    ),

    (
        "Viscosity_cP",
        "Viscosity (cP)",
        "Viscosity Response to Adulteration"
    ),

    (
        "Refractive_Index",
        "Refractive Index",
        "Refractive-Index Response to Adulteration"
    ),

    (
        "Dielectric_Constant",
        "Dielectric Constant",
        "Dielectric-Constant Response to Adulteration"
    )
]


for feature, ylabel, title in property_plot_definitions:

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(15, 4.8),
        sharex=True
    )

    for ax, fuel in zip(
        axes,
        BASE_FUELS
    ):

        fuel_data = dataset[
            dataset["Base_Fuel"] == fuel
        ]

        for adulterant in [
            x
            for x in adulterant_order
            if (fuel, x) in VALID_PAIRS
        ]:

            subset = fuel_data[
                fuel_data["Adulterant"]
                == adulterant
            ]

            summary = (
                subset
                .groupby("Adulteration_%")[feature]
                .agg(
                    ["mean", "std"]
                )
                .reset_index()
            )

            ax.plot(
                summary["Adulteration_%"],
                summary["mean"],
                marker="o",
                markersize=3.5,
                linewidth=1.7,
                label=adulterant,
                color=ADULTERANT_COLORS[
                    adulterant
                ]
            )

            ax.fill_between(
                summary["Adulteration_%"],
                summary["mean"]
                - summary["std"],
                summary["mean"]
                + summary["std"],
                color=ADULTERANT_COLORS[
                    adulterant
                ],
                alpha=0.10
            )

        ax.set_title(
            fuel,
            fontweight="bold"
        )

        ax.set_xlabel(
            "Adulteration concentration (%)"
        )

        ax.set_ylabel(
            ylabel
        )

        ax.set_xticks(
            NONZERO_LEVELS
        )

        ax.grid(
            True,
            alpha=0.25
        )

        sns.despine(ax=ax)

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.08),
        ncol=3,
        frameon=False,
        title="Adulterant"
    )

    fig.suptitle(
        title,
        fontsize=15,
        fontweight="bold",
        y=1.15
    )

    fig.tight_layout()

    safe_name = (
        feature
        .replace("_", "_")
    )

    save_research_figure(
        fig,
        f"Figure_07_{safe_name}_Response"
    )


# ============================================================
# 31. FIGURE 8 — COLOUR RESPONSE
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5),
    sharex=True
)

for ax, fuel in zip(
    axes,
    BASE_FUELS
):

    fuel_data = dataset[
        dataset["Base_Fuel"] == fuel
    ]

    valid_adulterants = [
        x
        for x in adulterant_order
        if (fuel, x) in VALID_PAIRS
    ]

    for adulterant in valid_adulterants:

        subset = fuel_data[
            fuel_data["Adulterant"]
            == adulterant
        ]

        summary = (
            subset
            .groupby("Adulteration_%")
            ["Colour_Value"]
            .agg(
                ["mean", "std"]
            )
            .reset_index()
        )

        ax.plot(
            summary["Adulteration_%"],
            summary["mean"],
            marker="o",
            markersize=3.5,
            linewidth=1.7,
            label=adulterant,
            color=ADULTERANT_COLORS[
                adulterant
            ]
        )

        ax.fill_between(
            summary["Adulteration_%"],
            summary["mean"]
            - summary["std"],
            summary["mean"]
            + summary["std"],
            color=ADULTERANT_COLORS[
                adulterant
            ],
            alpha=0.10
        )

    ax.set_title(
        fuel,
        fontweight="bold"
    )

    ax.set_xlabel(
        "Adulteration concentration (%)"
    )

    ax.set_ylabel(
        "Colour value"
    )

    ax.set_xticks(
        NONZERO_LEVELS
    )

    ax.grid(
        True,
        alpha=0.25
    )

    sns.despine(ax=ax)


handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.08),
    ncol=3,
    frameon=False,
    title="Adulterant"
)

fig.suptitle(
    "Colour Response to Petroleum Product Adulteration",
    fontsize=15,
    fontweight="bold",
    y=1.15
)

fig.tight_layout()

save_research_figure(
    fig,
    "Figure_08_Colour_Response"
)


# ============================================================
# 32. FIGURE 9 — PROPERTY BOX PLOTS BY BASE FUEL
# ============================================================

fig, axes = plt.subplots(
    2,
    3,
    figsize=(14, 9)
)

axes = axes.flatten()

box_features = [
    "Density_g_mL",
    "Viscosity_cP",
    "Refractive_Index",
    "Dielectric_Constant",
    "Colour_Value"
]

for i, feature in enumerate(
    box_features
):

    ax = axes[i]

    sns.boxplot(
        data=dataset,
        x="Base_Fuel",
        y=feature,
        order=BASE_FUELS,
        palette=FUEL_COLORS,
        width=0.55,
        fliersize=1.5,
        ax=ax
    )

    ax.set_title(
        feature_labels[feature]
    )

    ax.set_xlabel(
        "Base fuel"
    )

    ax.set_ylabel(
        feature_labels[feature]
    )

    sns.despine(ax=ax)

# Hide unused axis
axes[-1].axis("off")

fig.suptitle(
    "Distribution of Simulated Physical Properties by Base Fuel",
    fontsize=15,
    fontweight="bold",
    y=0.995
)

fig.tight_layout(
    rect=[0, 0, 1, 0.98]
)

save_research_figure(
    fig,
    "Figure_09_Property_Distributions_By_Fuel"
)


# ============================================================
# 33. FIGURE 10 — RANDOMIZED DATASET PREVIEW
# ============================================================
#
# This figure demonstrates that Sample_ID values no longer
# reveal a sequential experimental block.
# ============================================================

preview_n = 150

preview = dataset.head(
    preview_n
)

fig, ax = plt.subplots(
    figsize=(11, 5)
)

fuel_numeric = {
    "PMS": 1,
    "AGO": 2,
    "DPK": 3
}

y_values = [
    fuel_numeric[x]
    for x in preview["Base_Fuel"]
]

ax.scatter(
    preview["Sample_ID"],
    y_values,
    c=[
        FUEL_COLORS[x]
        for x in preview["Base_Fuel"]
    ],
    s=20,
    alpha=0.75
)

ax.set_yticks(
    [1, 2, 3]
)

ax.set_yticklabels(
    ["PMS", "AGO", "DPK"]
)

ax.set_xlabel(
    "Randomized Sample ID"
)

ax.set_ylabel(
    "Base fuel"
)

ax.set_title(
    "Randomized Ordering of Dataset Records"
)

ax.grid(
    True,
    alpha=0.20
)

sns.despine(ax=ax)

save_research_figure(
    fig,
    "Figure_10_Randomized_Record_Order"
)


# ============================================================
# 34. CREATE NUMERICAL SUMMARY REPORT
# ============================================================

numerical_summary = (
    dataset[
        [
            "Adulteration_%",
            "Base_Volume_mL",
            "Adulterant_Volume_mL",
            "Density_g_mL",
            "Viscosity_cP",
            "Refractive_Index",
            "Dielectric_Constant",
            "Colour_Value"
        ]
    ]
    .describe()
    .T
)

summary_file = (
    REPORTS_DIR
    / "numerical_summary.csv"
)

numerical_summary.to_csv(
    summary_file
)


# ============================================================
# 35. SAVE CORRELATION MATRIX DATA
# ============================================================

correlation_file = (
    REPORTS_DIR
    / "pearson_correlation_matrix.csv"
)

correlation_matrix.to_csv(
    correlation_file
)


# ============================================================
# 36. SAVE CONDITION FREQUENCY REPORT
# ============================================================

condition_frequency_file = (
    REPORTS_DIR
    / "condition_frequency.csv"
)

condition_summary.to_csv(
    condition_frequency_file,
    index=False
)


# ============================================================
# 37. SAVE DATASET INFORMATION TEXT
# ============================================================

dataset_information = f"""
PHYSICS-GUIDED PETROLEUM ADULTERATION DATASET
============================================================

Dataset size:
216,080 observations

Dataset type:
Physics-guided simulated raw dataset

Purpose:
Training/development of an Edge AI framework for petroleum
product adulteration detection.

Base fuels:
1. PMS - Premium Motor Spirit
2. AGO - Automotive Gas Oil
3. DPK - Dual Purpose Kerosene

Adulterants:
1. Kerosene
2. AGO
3. Water
4. Waste Engine Oil
5. Methanol
6. Ethanol

Excluded:
AGO + AGO
DPK + Kerosene

Concentration levels:
0%, 5%, 10%, 15%, 20%, 25%, 30%, 35%, 40%, 45%, 50%

Mixture volume:
200 mL

Raw physical features:
Density
Viscosity
Refractive Index
Dielectric Constant
Colour Value

No Label column:
This is intentional. The dataset remains a raw research dataset.
Classification and regression targets can be constructed during
the preprocessing stage.

Randomization:
The final records were shuffled after generation.
Sample_ID was assigned only after shuffling.

Therefore Sample_ID does not correspond to fuel type,
adulterant type or concentration order.

Gaussian noise:
Applied only to simulated physical properties.

Noise model:
X_sim = X_model * (1 + epsilon)

epsilon ~ N(0, sigma)

Noise assumptions:
Density = 0.20%
Viscosity = 1.00%
Refractive Index = 0.05%
Dielectric Constant = 1.50%
Colour = 1.00%

Scientific distinction:
The dataset is simulated and should not be described as laboratory
measurements. Actual laboratory measurements are intended to serve
as an independent validation dataset.

============================================================
"""

information_file = (
    DOCS_DIR
    / "README_DATASET.txt"
)

with open(
    information_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(dataset_information)


# ============================================================
# 38. CREATE FILE MANIFEST
# ============================================================

manifest = []

for root, dirs, files in os.walk(
    OUTPUT_ROOT
):

    for filename in files:

        file_path = (
            Path(root)
            / filename
        )

        manifest.append({
            "File":
                str(
                    file_path.relative_to(
                        OUTPUT_ROOT
                    )
                ),

            "Size_MB":
                round(
                    file_path.stat().st_size
                    / (1024 * 1024),
                    3
                )
        })

manifest_df = pd.DataFrame(
    manifest
)

manifest_file = (
    REPORTS_DIR
    / "file_manifest.csv"
)

manifest_df.to_csv(
    manifest_file,
    index=False
)


# ============================================================
# 39. CREATE ZIP PACKAGE
# ============================================================

zip_file = Path(
    "/content/"
    "physics_guided_petroleum_adulteration_dataset_FINAL.zip"
)

if zip_file.exists():
    zip_file.unlink()

with zipfile.ZipFile(
    zip_file,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:

    for root, dirs, files in os.walk(
        OUTPUT_ROOT
    ):

        for filename in files:

            file_path = (
                Path(root)
                / filename
            )

            arcname = (
                Path("physics_guided_petroleum_dataset")
                / file_path.relative_to(
                    OUTPUT_ROOT
                )
            )

            zipf.write(
                file_path,
                arcname
            )


# ============================================================
# 40. FINAL VALIDATION AFTER FILE CREATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

print(
    f"Rows: {len(dataset):,}"
)

print(
    f"Columns: {len(dataset.columns)}"
)

print(
    "Missing values:",
    int(dataset.isna().sum().sum())
)

print(
    "Unique Sample IDs:",
    dataset["Sample_ID"].nunique()
)

print(
    "Sample ID range:",
    dataset["Sample_ID"].min(),
    "to",
    dataset["Sample_ID"].max()
)

print(
    "\nBase-fuel distribution:"
)

print(
    dataset["Base_Fuel"]
    .value_counts()
    .sort_index()
)

print(
    "\nAdulteration distribution:"
)

print(
    dataset["Adulteration_%"]
    .value_counts()
    .sort_index()
)

print(
    "\nRaw columns:"
)

for column in dataset.columns:
    print(
        f"  - {column}"
    )


# ============================================================
# 41. OUTPUT LOCATIONS
# ============================================================

print("\n" + "=" * 70)
print("FILES CREATED")
print("=" * 70)

print(
    "\nMain CSV:"
)

print(csv_file)

print(
    "\nExcel workbook:"
)

print(excel_file)

print(
    "\nZIP package:"
)

print(zip_file)

print(
    "\nPlots:"
)

print(PLOTS_DIR)

print(
    "\nReports:"
)

print(REPORTS_DIR)

print(
    "\nDocumentation:"
)

print(DOCS_DIR)


# ============================================================
# 42. AUTOMATIC DOWNLOAD IN GOOGLE COLAB
# ============================================================

print("\nPreparing automatic download...")

try:

    from google.colab import files

    print(
        "\nGoogle Colab detected."
    )

    print(
        "Starting automatic ZIP download..."
    )

    files.download(
        str(zip_file)
    )

except ImportError:

    print(
        "\nGoogle Colab was not detected."
    )

    print(
        "The ZIP file has been created here:"
    )

    print(zip_file)


# ============================================================
# 43. COMPLETION MESSAGE
# ============================================================

print("\n" + "=" * 70)
print("DATASET GENERATION COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    f"\nFinal dataset: {len(dataset):,} observations"
)

print(
    "Final dataset order: RANDOMIZED"
)

print(
    "Sample IDs: ASSIGNED AFTER RANDOMIZATION"
)

print(
    "Raw dataset: NO LABEL COLUMN"
)

print(
    "Plots: HIGH-QUALITY 600-DPI PNG + VECTOR PDF"
)

print(
    "Correlation matrix: PEARSON, RESEARCH QUALITY"
)

print(
    "Package: READY"
)

print("=" * 70)

GENERATING PHYSICS-GUIDED PETROLEUM DATASET

Generating pure-fuel observations...
Pure observations generated: 76,080

Generating adulterated observations...
Adulterated observations generated: 140,000

Randomizing final row order...

Performing dataset integrity checks...
All integrity checks passed.

FINAL DATASET SUMMARY
Total observations : 216,080
Pure observations  : 76,080
Adulterated        : 140,000
Base fuels         : 3
Adulterants        : 6
Concentration levels: 11

CSV saved:
/content/physics_guided_petroleum_dataset/01_Dataset/physics_guided_petroleum_adulteration_raw_dataset.csv
Excel workbook saved:
/content/physics_guided_petroleum_dataset/01_Dataset/physics_guided_petroleum_adulteration_raw_dataset.xlsx


/tmp/ipykernel_1240/120357907.py:2221: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/tmp/ipykernel_1240/120357907.py:2221: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/tmp/ipykernel_1240/120357907.py:2221: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/tmp/ipykernel_1240/120357907.py:2221: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/tmp/ipykernel_1240/120357907.py:2221: FutureWarning


FINAL VALIDATION
Rows: 216,080
Columns: 11
Missing values: 0
Unique Sample IDs: 216080
Sample ID range: 1 to 216080

Base-fuel distribution:
Base_Fuel
AGO    69110
DPK    69110
PMS    77860
Name: count, dtype: int64

Adulteration distribution:
Adulteration_%
0     76080
5     14000
10    14000
15    14000
20    14000
25    14000
30    14000
35    14000
40    14000
45    14000
50    14000
Name: count, dtype: int64

Raw columns:
  - Sample_ID
  - Base_Fuel
  - Adulterant
  - Adulteration_%
  - Base_Volume_mL
  - Adulterant_Volume_mL
  - Density_g_mL
  - Viscosity_cP
  - Refractive_Index
  - Dielectric_Constant
  - Colour_Value

FILES CREATED

Main CSV:
/content/physics_guided_petroleum_dataset/01_Dataset/physics_guided_petroleum_adulteration_raw_dataset.csv

Excel workbook:
/content/physics_guided_petroleum_dataset/01_Dataset/physics_guided_petroleum_adulteration_raw_dataset.xlsx

ZIP package:
/content/physics_guided_petroleum_adulteration_dataset_FINAL.zip

Plots:
/content/physics_guid

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


DATASET GENERATION COMPLETED SUCCESSFULLY

Final dataset: 216,080 observations
Final dataset order: RANDOMIZED
Sample IDs: ASSIGNED AFTER RANDOMIZATION
Raw dataset: NO LABEL COLUMN
Plots: HIGH-QUALITY 600-DPI PNG + VECTOR PDF
Correlation matrix: PEARSON, RESEARCH QUALITY
Package: READY
